# Epidemic Systems Modeling: Compartmental Models for COVID-19
## Capstone Project Notebook

Compartmental models are mathematical frameworks used to study the spread of infectious diseases like COVID-19. They divide a population into distinct groups, or "compartments," based on disease status, such as Susceptible (S), Exposed (E), Infected (I), or Recovered (R). These models use differential equations to track how individuals move between compartments over time, driven by processes like infection, recovery, or interventions (e.g., vaccination).

The SIR and SEIR models are foundational examples. The **SIR model** tracks Susceptible, Infected, and Recovered individuals, while the **SEIR model** adds an Exposed compartment to account for the incubation period — a critical feature for diseases like COVID-19. These models are widely used in epidemiology to study complex dynamics, such as how social distancing or vaccination can "flatten the curve" or prevent multiple epidemic waves.

### Compartmental Models vs. Real-World Epidemiology

Compartmental models are *simplified* representations of disease spread. They focus on core processes (e.g., transmission, recovery) and assume a **well-mixed population** where everyone has an equal chance of interacting. In contrast, real-world epidemiology involves complex factors like spatial variation, heterogeneous contact patterns, specific viral strains, or detailed demographic data. Advanced models, such as agent-based simulations or network models, incorporate these details for more accurate predictions.

In this project, you will start with basic compartmental models (SIR and SEIR) and extend them to include COVID-19-specific features, such as intervention effects or age-structured populations, to bridge the gap toward more realistic scenarios.

### In this notebook you will:
1. Implement the **SIR** and **SEIR** compartmental models from scratch
2. Understand how each compartment and parameter maps to real COVID-19 biology
3. Calibrate parameters using **published literature values**
4. Simulate baseline epidemics and compare model structures
5. Explore **interventions** (lockdown, vaccination) within realistic bounds
6. Build a foundation for the required project extensions

---

## 0 · Setup

We import NumPy for numerical integration and Matplotlib for plotting. We also define a colour palette for the four compartments: green for Susceptible (healthy), orange for Exposed (incubating), red for Infected (contagious), and blue for Recovered (immune). These colours will be used consistently throughout the notebook.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.dpi': 120,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 12,
})

C_S = '#2ecc71'; C_E = '#f39c12'; C_I = '#e74c3c'; C_R = '#3498db'
print('Imports loaded successfully.')

---
## 1 · The SIR Model

The SIR model has three compartments:

- **Susceptible (S):** Individuals who can contract the disease.
- **Infected (I):** Individuals who are infectious and can spread the disease.
- **Recovered (R):** Individuals who have recovered and are immune (or removed, e.g., deceased).

The dynamics are governed by:
- **Infection:** A Susceptible individual becomes Infected at rate $\beta S I / N$, where $\beta$ is the transmission rate.
- **Recovery:** An Infected individual becomes Recovered at rate $\gamma I$, where $\gamma$ is the recovery rate.

$$\frac{dS}{dt} = -\beta \frac{SI}{N}, \qquad \frac{dI}{dt} = \beta \frac{SI}{N} - \gamma I, \qquad \frac{dR}{dt} = \gamma I$$

| Parameter | Meaning | COVID-19 estimate |
|-----------|---------|-------------------|
| $\beta$ | Transmission rate | $\approx 0.25\text{–}0.50$ day$^{-1}$ |
| $\gamma$ | Recovery rate ($1/\gamma$ = infectious period) | $\approx 0.1$ day$^{-1}$ (10-day illness) |
| $R_0 = \beta/\gamma$ | Basic reproduction number | $\approx 2.5\text{–}3.5$ for original strain |

### Key limitation for COVID-19
The SIR model assumes individuals become infectious **immediately** upon infection. In reality, COVID-19 has a 4–6 day **incubation period** where individuals are exposed but not yet infectious. This motivates the SEIR model.

Below we implement `sir_rhs` (the right-hand side of the ODE system) and `simulate_sir` (an Euler-method solver that steps through time). We work with **fractions** of the total population ($S + I + R = 1$) rather than absolute numbers. After the simulation we verify that the total population is conserved at every time step — a basic sanity check for any compartmental model.

In [ ]:
def sir_rhs(S, I, R, beta, gamma):
    """Rates of change for the SIR model."""
    dS = -beta * S * I
    dI = beta * S * I - gamma * I
    dR = gamma * I
    return dS, dI, dR

def simulate_sir(S0, I0, R0, beta, gamma, T, dt=0.1):
    """Euler simulation of the SIR model."""
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    S, I, R = np.zeros(steps), np.zeros(steps), np.zeros(steps)
    S[0], I[0], R[0] = S0, I0, R0
    for k in range(1, steps):
        dS, dI, dR = sir_rhs(S[k-1], I[k-1], R[k-1], beta, gamma)
        S[k] = S[k-1] + dt * dS
        I[k] = I[k-1] + dt * dI
        R[k] = R[k-1] + dt * dR
    return t, S, I, R

# Conservation test
t, S, I, R = simulate_sir(0.99, 0.01, 0.0, 0.3, 0.1, 200)
assert np.allclose(S + I + R, 1.0), 'Population not conserved!'
print(f'SIR test passed. Peak infected: {np.max(I):.3f} at day {t[np.argmax(I)]:.0f}')

---
## 2 · The SEIR Model

The SEIR model adds an **Exposed** compartment to capture the incubation period:

- **Exposed (E):** Individuals who are infected but not yet infectious (e.g., in the incubation period).

An Exposed individual becomes Infected at rate $\sigma E$, where $\sigma$ is the rate of progression (inverse of the incubation period).

$$\frac{dS}{dt} = -\beta \frac{SI}{N}, \quad \frac{dE}{dt} = \beta \frac{SI}{N} - \sigma E, \quad \frac{dI}{dt} = \sigma E - \gamma I, \quad \frac{dR}{dt} = \gamma I$$

| Parameter | Meaning | COVID-19 estimate |
|-----------|---------|-------------------|
| $\sigma$ | Rate of becoming infectious ($1/\sigma$ = latent period) | $\approx 0.2$ day$^{-1}$ (5-day incubation) |

### Why SEIR matters for COVID-19
- The 5-day incubation period **delays** the epidemic peak
- Exposed individuals are not yet infectious — this affects **contact tracing** strategies
- $R_0$ is the same ($\beta/\gamma$), but the **generation time** is longer: $1/\sigma + 1/\gamma$

### Why are compartmental models important?

These models, despite their simplicity, serve several critical purposes:
- **Predicting epidemic trajectories:** Models forecast peak infection rates, healthcare demands, or epidemic duration, aiding resource planning.
- **Evaluating interventions:** Simulations show how measures like vaccination, social distancing, or masking alter epidemic curves, guiding policy decisions.
- **Revealing key dynamics:** Models highlight critical factors, such as the role of asymptomatic transmission or the impact of early intervention.
- **Simplifying complex systems:** By focusing on core processes, models provide insights into dynamics like multiple epidemic waves or endemic states.

The code below implements the SEIR system in the same style as the SIR model. We now have four state variables, and the conservation law is $S + E + I + R = 1$. Notice that new infections flow into the Exposed compartment first, and only transition to Infectious after a delay of $\sim 1/\sigma$ days.

In [ ]:
def seir_rhs(S, E, I, R, beta, sigma, gamma):
    """Rates of change for the SEIR model."""
    dS = -beta * S * I
    dE = beta * S * I - sigma * E
    dI = sigma * E - gamma * I
    dR = gamma * I
    return dS, dE, dI, dR

def simulate_seir(S0, E0, I0, R0, beta, sigma, gamma, T, dt=0.1):
    """Euler simulation of the SEIR model."""
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    S, E, I, R = [np.zeros(steps) for _ in range(4)]
    S[0], E[0], I[0], R[0] = S0, E0, I0, R0
    for k in range(1, steps):
        dS, dE, dI, dR = seir_rhs(S[k-1], E[k-1], I[k-1], R[k-1],
                                   beta, sigma, gamma)
        S[k] = S[k-1] + dt * dS
        E[k] = E[k-1] + dt * dE
        I[k] = I[k-1] + dt * dI
        R[k] = R[k-1] + dt * dR
    return t, S, E, I, R

# Conservation test
t_se, S_se, E_se, I_se, R_se = simulate_seir(0.99, 0.0, 0.01, 0.0,
                                               0.3, 0.2, 0.1, 200)
total = S_se + E_se + I_se + R_se
assert np.allclose(total, 1.0), 'Population not conserved!'
print(f'SEIR test passed. Peak infected: {np.max(I_se):.3f} at day {t_se[np.argmax(I_se)]:.0f}')

---
## 3 · SIR vs SEIR: Structural Comparison

Now that we have both models implemented, we can compare them directly. We use identical parameters ($\beta = 0.3$, $\gamma = 0.1$) and initial conditions (1% initially infected) so that any differences are purely due to the **model structure** — the presence or absence of the Exposed compartment.

Key things to look for in the plots below:
- The **SEIR peak is delayed** compared to SIR because newly infected individuals spend time in the Exposed state before becoming infectious.
- The **SEIR peak is slightly lower** — the incubation delay gives the susceptible population more time to deplete before the infectious wave crests.
- The **final epidemic size** (total fraction recovered) is similar in both models, since $R_0$ is the same.
- The SEIR curve has a more **gradual rise** — the exposed compartment acts as a buffer that smooths the transition.

In [ ]:
beta, gamma, sigma = 0.3, 0.1, 0.2
R0_val = beta / gamma

t_sir, S_sir, I_sir, R_sir = simulate_sir(0.99, 0.01, 0.0, beta, gamma, 200)
t_seir, S_seir, E_seir, I_seir, R_seir = simulate_seir(0.99, 0.0, 0.01, 0.0,
                                                         beta, sigma, gamma, 200)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

ax1.plot(t_sir, S_sir, color=C_S, lw=2, label='S')
ax1.plot(t_sir, I_sir, color=C_I, lw=2, label='I')
ax1.plot(t_sir, R_sir, color=C_R, lw=2, label='R')
ax1.set_title('SIR Model', fontweight='bold')
ax1.set_xlabel('Days'); ax1.set_ylabel('Fraction'); ax1.legend()

ax2.plot(t_seir, S_seir, color=C_S, lw=2, label='S')
ax2.plot(t_seir, E_seir, color=C_E, lw=2, label='E')
ax2.plot(t_seir, I_seir, color=C_I, lw=2, label='I')
ax2.plot(t_seir, R_seir, color=C_R, lw=2, label='R')
ax2.set_title('SEIR Model', fontweight='bold')
ax2.set_xlabel('Days'); ax2.legend()

fig.suptitle(f'SIR vs SEIR ($R_0 = {R0_val:.1f}$, latent period = {1/sigma:.0f} days)',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.show()

print(f'SIR  peak: {np.max(I_sir):.3f} at day {t_sir[np.argmax(I_sir)]:.0f}')
print(f'SEIR peak: {np.max(I_seir):.3f} at day {t_seir[np.argmax(I_seir)]:.0f}')
print('The SEIR peak is delayed and slightly lower due to the incubation period.')

---
## 4 · COVID-19 Baseline Simulation

Let us use literature-based parameters for the original SARS-CoV-2 strain:

| Parameter | Value | Source |
|-----------|-------|--------|
| $R_0$ | 2.5 | Li et al. (2020), NEJM |
| Infectious period $1/\gamma$ | 10 days | WHO situation reports |
| Incubation period $1/\sigma$ | 5.2 days | Lauer et al. (2020), Ann. Intern. Med. |
| $\beta = R_0 \cdot \gamma$ | 0.25 day$^{-1}$ | Derived |

> **Important:** These are estimates with uncertainty. Real values varied by region, time, and variant.

We simulate a full year (365 days) starting from a tiny seed of infection (0.01% of the population). This represents the early introduction of the virus into a fully susceptible population with no interventions — the "worst case" unmitigated baseline. The simulation prints the peak timing, peak fraction infected, and the total attack rate (fraction eventually infected).

In [ ]:
# COVID-19 literature-based parameters
gamma_covid = 1 / 10.0      # 10-day infectious period
sigma_covid = 1 / 5.2       # 5.2-day incubation period
R0_covid = 2.5
beta_covid = R0_covid * gamma_covid

print(f'Parameters: beta={beta_covid:.3f}, sigma={sigma_covid:.3f}, gamma={gamma_covid:.3f}')
print(f'R0 = {beta_covid/gamma_covid:.1f}')
print(f'Generation time ≈ {1/sigma_covid + 1/gamma_covid:.1f} days')

# Simulate 365 days, starting with 0.01% infected
t_cv, S_cv, E_cv, I_cv, R_cv = simulate_seir(
    S0=1-1e-4, E0=0, I0=1e-4, R0=0,
    beta=beta_covid, sigma=sigma_covid, gamma=gamma_covid, T=365)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(t_cv, S_cv, color=C_S, lw=2, label='Susceptible')
ax.plot(t_cv, E_cv, color=C_E, lw=2, label='Exposed')
ax.plot(t_cv, I_cv, color=C_I, lw=2, label='Infectious')
ax.plot(t_cv, R_cv, color=C_R, lw=2, label='Recovered')
ax.set_xlabel('Days since first case')
ax.set_ylabel('Fraction of population')
ax.set_title(f'SEIR COVID-19 Baseline ($R_0 = {R0_covid}$, no interventions)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

peak_day = t_cv[np.argmax(I_cv)]
peak_frac = np.max(I_cv)
final_infected = R_cv[-1]
print(f'Peak infected: {peak_frac:.1%} at day {peak_day:.0f}')
print(f'Total infected by day 365: {final_infected:.1%}')
print(f'Herd immunity threshold: {1-1/R0_covid:.1%}')

---
## 5 · Sensitivity Analysis: Varying $R_0$

The basic reproduction number $R_0$ is the single most important parameter in any epidemic model — it determines whether an outbreak grows ($R_0 > 1$) or dies out ($R_0 < 1$). For COVID-19, $R_0$ varied substantially across variants:

- **Original strain** — $R_0 \approx 2.5$
- **Alpha variant** — $R_0 \approx 3.5\text{–}4.0$
- **Delta variant** — $R_0 \approx 4.0\text{–}6.0$
- **Omicron variant** — $R_0 \approx 6.0\text{–}10.0$

Below we simulate four values of $R_0$ while holding the infectious period and incubation period constant. Higher $R_0$ produces a faster, higher, and earlier peak — and infects a larger fraction of the population overall. Note how even a modest increase from 2.5 to 4.0 dramatically changes the epidemic curve.

In [ ]:
R0_values = [1.5, 2.5, 4.0, 6.0]
labels = ['Low (1.5)', 'Original (2.5)', 'Delta-like (4.0)', 'Omicron-like (6.0)']
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#8e44ad']

fig, ax = plt.subplots(figsize=(10, 5))
for R0_v, lab, col in zip(R0_values, labels, colors):
    beta_v = R0_v * gamma_covid
    t_v, _, _, I_v, _ = simulate_seir(1-1e-4, 0, 1e-4, 0,
                                       beta_v, sigma_covid, gamma_covid, 365)
    ax.plot(t_v, I_v, color=col, lw=2, label=f'{lab}: peak {np.max(I_v):.1%}')

ax.set_xlabel('Days')
ax.set_ylabel('Infectious fraction')
ax.set_title('Effect of $R_0$ on Epidemic Dynamics', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

---
## 6 · Modeling Interventions

### Lockdown: reducing the contact rate

A lockdown reduces $\beta$ by decreasing the contact rate. Realistic assumptions:
- A strict lockdown might reduce contacts by 60–80% (not 100%)
- Compliance is imperfect and varies over time

We model this as a time-dependent $\beta(t)$: during the lockdown window, $\beta_\text{eff} = \beta \cdot (1 - \text{reduction})$. Outside that window, $\beta$ returns to its baseline value.

The function `simulate_seir_intervention` also supports a simple **vaccination** mechanism: from a given start day onward, a fixed fraction of the susceptible population is moved directly to the Recovered compartment each day, representing immunisation.

Below we compare three lockdown scenarios — starting at day 30, 60, or 90 — each lasting 60 days with a 70% contact reduction. The key insight: **earlier lockdowns are far more effective** at reducing the peak, but the epidemic can rebound once restrictions are lifted if enough susceptibles remain. This "exit problem" was a central challenge in real-world COVID-19 policy.

In [ ]:
def simulate_seir_intervention(S0, E0, I0, R0, beta_base, sigma, gamma,
                                T, dt=0.1,
                                lockdown_start=None, lockdown_end=None,
                                lockdown_reduction=0.0,
                                vacc_start=None, vacc_rate=0.0):
    """SEIR with optional lockdown and vaccination.
    
    lockdown_reduction: fraction by which beta is reduced (0 to 1)
    vacc_rate: fraction of S vaccinated per day (moved to R)
    """
    steps = int(T / dt) + 1
    t = np.linspace(0, T, steps)
    S, E, I, R = [np.zeros(steps) for _ in range(4)]
    S[0], E[0], I[0], R[0] = S0, E0, I0, R0
    
    for k in range(1, steps):
        day = t[k-1]
        # Lockdown effect
        beta_eff = beta_base
        if lockdown_start is not None and lockdown_end is not None:
            if lockdown_start <= day <= lockdown_end:
                beta_eff = beta_base * (1 - lockdown_reduction)
        
        dS, dE, dI, dR = seir_rhs(S[k-1], E[k-1], I[k-1], R[k-1],
                                   beta_eff, sigma, gamma)
        
        # Vaccination effect
        vacc = 0
        if vacc_start is not None and day >= vacc_start:
            vacc = min(vacc_rate * dt, S[k-1])  # can't vaccinate more than available
        
        S[k] = S[k-1] + dt * dS - vacc
        E[k] = E[k-1] + dt * dE
        I[k] = I[k-1] + dt * dI
        R[k] = R[k-1] + dt * dR + vacc
    
    return t, S, E, I, R

# Lockdown scenarios
fig, ax = plt.subplots(figsize=(10, 5))

# No intervention
t0, _, _, I0_curve, _ = simulate_seir_intervention(
    1-1e-4, 0, 1e-4, 0, beta_covid, sigma_covid, gamma_covid, 365)
ax.plot(t0, I0_curve, 'k-', lw=2, label='No intervention')

# Lockdown at day 30, 60 days, 70% reduction
for start, col in [(30, '#e74c3c'), (60, '#f39c12'), (90, '#3498db')]:
    t_ld, _, _, I_ld, _ = simulate_seir_intervention(
        1-1e-4, 0, 1e-4, 0, beta_covid, sigma_covid, gamma_covid, 365,
        lockdown_start=start, lockdown_end=start+60, lockdown_reduction=0.7)
    ax.plot(t_ld, I_ld, color=col, lw=2, ls='--',
            label=f'Lockdown day {start}–{start+60} (70% reduction)')

ax.set_xlabel('Days')
ax.set_ylabel('Infectious fraction')
ax.set_title('Effect of Lockdown Timing', fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

print('Earlier lockdowns are more effective at reducing the peak.')
print('But: once the lockdown ends, the epidemic can rebound if enough susceptibles remain.')

---
## 7 · Your Tasks

The code above provides baseline SIR and SEIR implementations with interventions. Your capstone project requires you to go substantially further.

### Required

1. **Model Implementation**: Implement at least two different compartmental models (e.g., SIR vs SEIR). Discuss why a basic model fails to capture realistic COVID-19 dynamics.

2. **Empirical Calibration**: Establish baseline parameters using **peer-reviewed literature**. You must cite real-world data for your initial values.

3. **Plausible Simulations**: Design intervention scenarios within **realistic bounds** (e.g., a lockdown cannot reduce contact to zero; vaccines are not 100% effective).

4. **Visualisation**: Show how interventions alter infection curves compared to the baseline.

### Extensions to implement

- **Sensitivity analysis**: Vary parameters within biologically plausible ranges. Show which parameters the model is most sensitive to.
- **Vaccination rollout**: Model a gradual vaccination campaign (e.g., 0.5% of population per day starting at day 90). Include realistic vaccine efficacy (e.g., 80–95%).
- **Multiple waves**: Show how relaxing restrictions leads to resurgence. Can you reproduce a two-wave pattern?
- **Age-structured model**: Split the population into age groups with different contact rates and severity.
- **Comparison with real data**: Fit your model to real COVID-19 case data from a specific country.

### Discussion points for your report

- Justify your model choice and parameter values with citations
- Analyse sensitivity within biologically plausible ranges
- Discuss limitations: spatial heterogeneity, behavioural changes, viral mutation
- Reflect on the challenge of matching clean mathematics to messy real-world data

In [ ]:
# ============================================================
# PLACEHOLDER: Implement your extensions below
# ============================================================

# Sensitivity analysis
# TODO: Vary beta, gamma, sigma within realistic ranges

# Vaccination rollout
# TODO: Gradual vaccination with realistic efficacy

# Multiple waves
# TODO: Lockdown → relaxation → resurgence

# Real data comparison
# TODO: Load real COVID data, fit model

---
## Recommended Reading & Journal Club

### Foundational References

**1. Kermack, W. O. & McKendrick, A. G. (1927)**
*A contribution to the mathematical theory of epidemics.*
Proc. Royal Society A, 115(772), 700–721. [DOI](https://doi.org/10.1098/rspa.1927.0118)
→ The original SIR model paper. Still worth reading for the elegance of the threshold theorem.

**2. Li, Q. et al. (2020)**
*Early transmission dynamics in Wuhan, China, of novel coronavirus–infected pneumonia.*
New England Journal of Medicine, 382(13), 1199–1207. [DOI](https://doi.org/10.1056/NEJMoa2001316)
→ Early R₀ estimates for SARS-CoV-2. Essential source for parameter calibration.

**3. Lauer, S. A. et al. (2020)**
*The incubation period of coronavirus disease 2019 (COVID-19) from publicly reported confirmed cases.*
Annals of Internal Medicine, 172(9), 577–582. [DOI](https://doi.org/10.7326/M20-0504)
→ Source for the 5.2-day incubation period estimate used in the SEIR model.

---

### Journal Club Papers

**4. Giordano, G. et al. (2020)**
*Modelling the COVID-19 epidemic and implementation of population-wide interventions in Italy.*
Nature Medicine, 26(6), 855–860. [DOI](https://doi.org/10.1038/s41591-020-0883-7)
→ Extended SEIR model (SIDARTHE) with diagnosed, ailing, threatened, and healed compartments. Example of a more realistic model.

**5. Flaxman, S. et al. (2020)**
*Estimating the effects of non-pharmaceutical interventions on COVID-19 in Europe.*
Nature, 584(7820), 257–261. [DOI](https://doi.org/10.1038/s41586-020-2405-7)
→ Bayesian estimation of intervention effects across 11 European countries.

**6. Prem, K. et al. (2020)**
*The effect of control strategies to reduce social mixing on outcomes of the COVID-19 epidemic in Wuhan, China.*
The Lancet Public Health, 5(5), e261–e270. [DOI](https://doi.org/10.1016/S2468-2667(20)30073-6)
→ Age-structured contact matrices and the impact of school closures and workplace restrictions.